# 03 — Li, Liu, Liu & Wei (2024), "Replicating and digesting anomalies in the Chinese A-share market"

All 469 anomalies, four portfolio procedures, seven factor models (lighthouse.md 5.3). Logic lives in `src/replication/li/`; this notebook only calls it and shows ours vs the paper. Run `python -m replication.li --build` first to fill the anomaly caches (aggregates only).

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 200)
from replication.li import tables as T, catalog as C
from replication.li.__main__ import TABLE_FUNCS
from replication.common.compare import compare
segs = T.li_segments(); orig = segs['original']

In [2]:
ours = pd.concat([f(orig) for f in TABLE_FUNCS.values()], ignore_index=True)
cmp = compare('li', ours)
scored = cmp[cmp['pass'].notna()]
summary = scored.groupby('table')['pass'].agg(['size', 'sum'])
summary.columns = ['scored', 'within tolerance']
summary

,scored,within tolerance
table,,
T1,530,523
T2,208,133
T3,168,53
T4,280,148
T5,392,148
T6,2224,1343
T7,448,144
TXT,20,9


In [3]:
def show(table, rows=None):
    x = cmp[cmp['table'] == table][['row', 'col', 'paper', 'ours', 'kind', 'pass']].copy()
    if rows: x = x[x['row'].str.match(rows)]
    x['pass'] = x['pass'].map({True: 'ok', False: 'MISS'}).fillna('')
    return x

## Table 1: sample and factors

In [4]:
show('T1', r'A:(1999|2005|2018)')

,row,col,paper,ours,kind,pass
0,A:1999,n_ashares,844.00,844.000000,count,ok
1,A:1999,n_sh_main,434.00,434.000000,count,ok
2,A:1999,n_sz_main,410.00,410.000000,count,ok
3,A:1999,n_sme,0.00,0.000000,count,ok
4,A:1999,n_gem,0.00,0.000000,count,ok
5,A:1999,tmv,2.40,2.404137,rel10,ok
6,A:1999,cmv,0.73,0.728271,rel10,ok
7,A:1999,sh_tmv,1.32,1.320716,rel10,ok
8,A:1999,sh_cmv,0.39,0.385134,rel10,ok
9,A:1999,sz_tmv,1.08,1.083421,rel10,ok


In [5]:
show('T1', r'[BC]:')

,row,col,paper,ours,kind,pass
300,B:MKT,mean,0.62,0.618148,pct_return,ok
301,B:ME,mean,0.78,0.774704,pct_return,ok
302,B:I/A,mean,0.29,0.263555,pct_return,ok
303,B:ROE,mean,0.54,0.489894,pct_return,ok
304,B:SMB-FF5,mean,0.68,0.678354,pct_return,ok
305,B:HML-FF5,mean,0.42,0.324150,pct_return,ok
306,B:RMW,mean,-0.14,-0.263670,pct_return,MISS
307,B:CMA,mean,0.05,0.077460,pct_return,ok
308,B:MKT-CH,mean,0.46,0.456404,pct_return,ok
309,B:SMB-CH3,mean,0.73,0.725356,pct_return,ok


## Table 2: microcaps and investment capacity

In [6]:
show('T2', r'A:')

,row,col,paper,ours,kind,pass
534,A:Market,n_firms,1563.00,1544.381579,count,ok
535,A:Market,pct_tmv,100.00,100.000000,share_pct,ok
536,A:Market,pct_cmv,100.00,100.000000,share_pct,ok
537,A:Market,vw_mean,0.80,0.742076,pct_return,ok
538,A:Market,vw_std,8.63,8.531548,pct_std,ok
539,A:Market,ew_mean,1.28,1.294907,pct_return,ok
540,A:Market,ew_std,9.84,9.736052,pct_std,MISS
541,A:Market,xs_std,9.88,11.010309,pct_std,MISS
542,A:Micro,n_firms,449.00,463.574561,count,MISS
543,A:Micro,pct_tmv,8.78,9.292303,share_pct,ok


In [7]:
show('T2', r'[BCD]:Mainboard-VW')

,row,col,paper,ours,kind,pass
574,B:Mainboard-VW,Low-All,8.20,7.681686,share_pct,ok
575,B:Mainboard-VW,Low-Mom,6.87,7.067130,share_pct,ok
576,B:Mainboard-VW,Low-VvG,6.87,8.730241,share_pct,ok
577,B:Mainboard-VW,Low-Inv,8.40,7.603930,share_pct,ok
578,B:Mainboard-VW,Low-Prof,11.94,9.724257,share_pct,ok
579,B:Mainboard-VW,Low-Intan,7.60,6.340305,share_pct,ok
580,B:Mainboard-VW,Low-Fric,6.15,6.721874,share_pct,ok
581,B:Mainboard-VW,High-All,6.80,6.662774,share_pct,ok
582,B:Mainboard-VW,High-Mom,6.25,4.101830,share_pct,ok
583,B:Mainboard-VW,High-VvG,5.56,8.141011,share_pct,ok


## Table 3: significance rates

In [8]:
show('T3', r'A:')

,row,col,paper,ours,kind,pass
742,A:Mainboard-VW,Momentum:SHT,15.56,35.555556,share_pct,MISS
743,A:Mainboard-VW,Momentum:MHT,4.44,8.888889,share_pct,MISS
744,A:Mainboard-VW,Value-versus-growth:SHT,8.82,2.941176,share_pct,MISS
745,A:Mainboard-VW,Value-versus-growth:MHT,0.00,1.470588,share_pct,ok
746,A:Mainboard-VW,Investment:SHT,2.78,5.555556,share_pct,ok
747,A:Mainboard-VW,Investment:MHT,0.00,2.777778,share_pct,ok
748,A:Mainboard-VW,Profitability:SHT,8.51,24.468085,share_pct,MISS
749,A:Mainboard-VW,Profitability:MHT,5.32,4.255319,share_pct,ok
750,A:Mainboard-VW,Intangibles:SHT,12.05,17.073171,share_pct,MISS
751,A:Mainboard-VW,Intangibles:MHT,2.41,3.658537,share_pct,ok


## Table 4: significant anomalies (Mainboard-VW)

In [9]:
show('T4', r'A:count')

,row,col,paper,ours,kind,pass
2314,A:count,significant,78.0,85.0,count,MISS


In [10]:
t4 = show('T4', r'A:'); t4[t4['col'] == 'm'].head(80)

,row,col,paper,ours,kind,pass
910,A:sue_q_1,m,1.17,1.043500,pct_return,MISS
928,A:sue_q_3,m,0.82,0.665778,pct_return,MISS
946,A:sue_q_6,m,0.54,0.469435,pct_return,ok
964,A:sue_q_9,m,0.47,0.371035,pct_return,ok
982,A:abr_q_9,m,0.32,0.281278,pct_return,ok
1000,A:_6_res_mom_3,m,0.56,0.609652,pct_return,ok
1018,A:_6_res_mom_6,m,0.44,0.483272,pct_return,ok
1036,A:ep_q_1,m,0.84,1.025108,pct_return,MISS
1054,A:ndp,m,0.68,0.311816,pct_return,MISS
1072,A:ndp_q_3,m,0.74,0.385918,pct_return,MISS


## Table 5: factor models

In [11]:
show('T5', r'A:')

,row,col,paper,ours,kind,pass
3414,A:CAPM,All:abs_alpha_hl,0.85,0.682913,pct_return,MISS
3415,A:CAPM,All:n_t196,74.00,80.000000,count,MISS
3416,A:CAPM,All:mean_abs_alpha,0.39,0.261917,pct_return,MISS
3417,A:CAPM,All:n_grs_p05,65.00,50.000000,count,MISS
3418,A:CAPM,Momentum:abs_alpha_hl,0.63,0.554035,pct_return,ok
3419,A:CAPM,Momentum:n_t196,7.00,16.000000,count,MISS
3420,A:CAPM,Momentum:mean_abs_alpha,0.32,0.225677,pct_return,ok
3421,A:CAPM,Momentum:n_grs_p05,4.00,13.000000,count,MISS
3422,A:CAPM,Value-versus-growth:abs_alpha_hl,0.72,1.025305,pct_return,MISS
3423,A:CAPM,Value-versus-growth:n_t196,4.00,2.000000,count,MISS


## Table 6 (excerpt) and Table 7

In [12]:
show('T6', r'A:sue_q_1')

,row,col,paper,ours,kind,pass
912,A:sue_q_1,alpha_FF5,1.11,1.228766e+00,pct_return,MISS
913,A:sue_q_1,t_FF5,4.36,5.159891e+00,t,ok
914,A:sue_q_1,mean_abs_alpha_FF5,0.38,3.435852e-01,pct_return,ok
915,A:sue_q_1,grs_p_FF5,0.00,5.287410e-07,pvalue,ok
916,A:sue_q_1,alpha_q,0.78,7.277164e-01,pct_return,ok
917,A:sue_q_1,t_q,3.10,3.200085e+00,t,ok
918,A:sue_q_1,mean_abs_alpha_q,0.21,2.177489e-01,pct_return,ok
919,A:sue_q_1,grs_p_q,0.07,8.944321e-03,pvalue,MISS
920,A:sue_q_1,alpha_CH3,0.47,4.038181e-01,pct_return,ok
921,A:sue_q_1,t_CH3,1.70,1.573110e+00,t,ok


In [13]:
show('T7', r'[AC]:Total')

,row,col,paper,ours,kind,pass
3902,A:Total,0407:Raw,44.0,44.0,count,ok
3903,A:Total,0407:CAPM,40.0,41.0,count,ok
3904,A:Total,0407:FF3,33.0,36.0,count,MISS
3905,A:Total,0407:FF5,36.0,38.0,count,MISS
3906,A:Total,0407:q,24.0,19.0,count,MISS
3907,A:Total,0407:CH3,20.0,14.0,count,MISS
3908,A:Total,0407:CH4,24.0,10.0,count,MISS
3909,A:Total,0407:Car4,39.0,37.0,count,MISS
3910,A:Total,0807:Raw,44.0,51.0,count,MISS
3911,A:Total,0807:CAPM,40.0,49.0,count,MISS


## Text numbers and section 4.2

In [14]:
show('TXT')

,row,col,paper,ours,kind,pass
4254,abstract:failure_rate_raw,pct,83.37,81.236203,share_pct,ok
4255,abstract:failure_rate_capm,pct,84.22,82.339956,share_pct,ok
4256,abstract:failure_rate_ff3,pct,86.99,84.768212,share_pct,ok
4257,S1:sig_full_sht,n,78.00,85.000000,count,MISS
4258,S1:sig_full_mht,n,29.00,18.000000,count,MISS
4259,S1:sig_post2007_sht,n,61.00,60.000000,count,ok
4260,S1:sig_post2007_mht,n,22.00,18.000000,count,MISS
4261,S1:explained_full_CH3,pct,53.85,75.294118,share_pct,MISS
4262,S1:explained_full_CH4,pct,47.44,74.117647,share_pct,MISS
4263,S1:explained_full_q,pct,25.64,54.117647,share_pct,MISS


## Extension (lighthouse 6): same functions, later years

The extended and full segments headline the look-ahead-free version (`portfolios.HOLDING_MONTH_FILTER = False`, user decision 2026-10-11); the authors' procedure is shown next to it.

In [ ]:
from csmar.registry import PROJECT_ROOT
R = PROJECT_ROOT / 'results' / 'li'
def headline(seg, variant):
    t = pd.read_csv(R / f"{seg}_tables{'' if variant == 'authors' else '_clean'}.csv")
    g = lambda tb, r, c: t[(t['table'] == tb) & (t['row'] == r) & (t['col'] == c)]['value'].iloc[0]
    return {'segment': seg, 'variant': variant,
            'share significant SHT (%)': round(g('T3', 'A:Mainboard-VW', 'All categories:SHT'), 2),
            'share significant MHT (%)': round(g('T3', 'A:Mainboard-VW', 'All categories:MHT'), 2),
            'n significant': int(g('T4', 'A:count', 'significant')),
            'VMG mean (%)': round(g('T1', 'B:VMG', 'mean'), 2) if len(t[(t['table']=='T1')&(t['row']=='B:VMG')]) else None}
# Headline per lighthouse 6 and the user's decision: authors' procedure for the original segment,
# look-ahead-free ("clean") for the extended and full segments; the other variant alongside.
pd.DataFrame([headline(s, v) for s in ('original', 'extended', 'full') for v in (('authors', 'clean') if s == 'original' else ('clean', 'authors'))])

### Extended and full segments (look-ahead-free headline): significance by category and factor-model summary

In [ ]:
for seg in ('extended', 'full'):
    t = pd.read_csv(R / f'{seg}_tables_clean.csv')
    x = t[(t['table'] == 'T3') & (t['row'] == 'A:Mainboard-VW')].set_index('col')['value'].round(2)
    print(seg); display(x.to_frame().T)
    t5 = t[(t['table'] == 'T5') & t['row'].str.startswith('A:') & t['col'].str.startswith('All:')]
    display(t5.pivot(index='row', columns='col', values='value').round(2))

## Month-by-month checks against the authors' series (lighthouse 3.3)

In [ ]:
fc = pd.read_csv(PROJECT_ROOT / 'results' / 'li_factor_check.csv'); display(fc.round(3))
gc = pd.read_csv(PROJECT_ROOT / 'results' / 'li_g5g1_check.csv')
print(gc['status'].value_counts().to_dict())
gc[gc['status'] == 'miss'].sort_values('corr').round(3)